<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-11/NB11_machine_consciousness.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 11: Machine Consciousness: Theories and Indicator Properties

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-11/lab.html>.

## Overview

Scientific theories of consciousness make claims about the mechanisms that give rise to experience. This week examines the global workspace, integrated information, higher-order and recurrent processing theories, and a recent method that derives indicator properties from them to assess AI systems [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to describe four major theories of consciousness and what each implies for machines, to compute a simplified integration measure for small networks, to explain the indicator property method and its assumption of computational functionalism, and to state the main disagreements about conscious AI.

## Global workspace theories

Baars proposed that consciousness corresponds to a global workspace: a limited-capacity stage on which information is broadcast to many specialised, unconscious processors [1]. Only content that wins the competition for the workspace becomes conscious and available for report and flexible use. Dehaene and Naccache developed a neuronal version in which conscious access involves a sudden, self-sustaining ignition of activity in long-range cortical networks [4]. On this family of theories, a machine with specialised modules, a limited workspace and global broadcast would satisfy the central conditions. The notebook simulates ignition, a nonlinear transition from local to global activity as stimulus strength increases.

## Integrated information theory

Tononi proposed that consciousness corresponds to integrated information, the extent to which a system as a whole generates information over and above its parts [2]. The theory was later formulated from axioms about the essential properties of experience, from which it derives requirements for its physical substrate [5, 6]. Its measure, Φ, is defined over the cause-effect structure of a system, and purely feedforward systems have zero Φ, whatever they compute. Integrated information theory therefore implies that a conventional digital computer running any program might have very little consciousness, because the relevant quantity depends on the physical causal structure, not on the software.

The lab and the notebook compute a much simpler quantity: how much the whole system's past predicts its future beyond what the best split into two parts predicts. Figure 11.1 shows that this simplified measure is zero for independent or feedforward networks and positive for recurrent loops, which mirrors one qualitative claim of the theory without reproducing its formal machinery.

![Figure 11.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-11/figures/w11_fig1.png)

*Figure 11.1. A simplified integration measure, whole-system information minus the information of the best split into parts, for five three-node boolean networks. It is inspired by, but not equal to, Φ of integrated information theory [2, 6].*

## Higher-order and recurrent processing theories

Higher-order theories hold that a mental state is conscious when the subject is aware of being in it, for example through a higher-order thought about the state [7]. A machine would need to represent some of its own states as its own. Recurrent processing theory holds that consciousness arises when feedback from higher to lower visual areas produces sustained recurrent activity, even without global broadcast [8]. The theories disagree about where consciousness begins, which matters for machines because different architectures satisfy different conditions.

## Indicator properties and the debate about AI

Butlin and colleagues proposed an empirical method for assessing AI systems [3]. Assuming computational functionalism as a working hypothesis, they derived indicator properties from several scientific theories, including recurrent processing, global workspace, higher-order, predictive processing and attention schema theories, together with agency and embodiment. They assessed several recent systems and concluded that no current AI system is conscious, but also that there are no obvious barriers to building systems that satisfy the indicators. Chalmers examined large language models and argued that they lack several features that many theories require, such as recurrent processing, a global workspace and unified agency, while such features could be added in future systems [9].

Others doubt the functionalist assumption. Seth argued for biological naturalism, the view that consciousness may depend on properties of living systems, such as the regulation of a living body, so that computation alone would not suffice [10]. The lab lets students weigh the theories by credence and see how the verdict about an architecture depends on those weights.

> **Pause and reflect.** Would you change how you treat a system if its indicator score rose? What does your answer reveal about the role of theory in such judgements?

# Hands-on lab

The notebook implements the simplified integration measure for small boolean networks and simulates ignition in a single global workspace unit with self-excitation [2, 4].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A simplified integration measure

With every state equally likely, the measure takes the mutual information between the whole system's state at one step and the next, subtracts the same quantity for each part of a bipartition, and keeps the weakest split. It is not Φ of integrated information theory, which is defined over cause-effect structures [6].

In [ ]:
import itertools

def mutual_information(states, nxt, idx):
    joint, px, py = {}, {}, {}
    w = 1 / len(states)
    for s, t in zip(states, nxt):
        a, b = tuple(s[i] for i in idx), tuple(t[i] for i in idx)
        joint[(a, b)] = joint.get((a, b), 0) + w
        px[a] = px.get(a, 0) + w
        py[b] = py.get(b, 0) + w
    return sum(p * np.log2(p / (px[a] * py[b])) for (a, b), p in joint.items())

def integration(update, n=3):
    states = list(itertools.product([0, 1], repeat=n))
    nxt = [tuple(update(s)) for s in states]
    whole = mutual_information(states, nxt, range(n))
    splits = []
    for k in range(1, n // 2 + 1):
        for A in itertools.combinations(range(n), k):
            B = tuple(i for i in range(n) if i not in A)
            splits.append(whole - mutual_information(states, nxt, A) - mutual_information(states, nxt, B))
    return max(0.0, min(splits))

NETWORKS = {
    "independent self-loops": lambda s: (s[0], s[1], s[2]),
    "feedforward chain": lambda s: (s[0], s[0], s[1]),
    "loop": lambda s: (s[2], s[0], s[1]),
}
for name, f in NETWORKS.items():
    print(f"{name:24s} {integration(f):.2f} bits")

### Your turn, exercise 1

Define the XOR loop, in which each node becomes the XOR of the other two, and compute its integration.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _xor_reference():
    return integration(lambda s: (s[1] ^ s[2], s[0] ^ s[2], s[0] ^ s[1]))

In [ ]:
xor_integration = None  # your computation
check("Exercise 1", xor_integration, _xor_reference())

## 2. Ignition in a global workspace unit

A workspace unit with strong self-excitation receives a stimulus of strength s. Above a critical strength, activity jumps to a high, self-sustaining state.

In [ ]:
def workspace(s, w=6.0, theta=3.0, temp=0.3, steps=400, dt=0.05):
    a = 0.0
    for _ in range(steps):
        a += dt * (-a + 1 / (1 + np.exp(-(w * a + s - theta) / temp)))
    return a

strengths = np.linspace(0, 3, 61)
final = [workspace(s) for s in strengths]
plt.plot(strengths, final, marker="o", ms=3)
plt.xlabel("stimulus strength"); plt.ylabel("final workspace activity"); plt.title("Nonlinear ignition"); plt.show()

### Your turn, exercise 2

Find the smallest stimulus strength in `strengths` for which the final activity exceeds 0.5. Explain in two sentences how this ignition relates to the distinction between conscious and unconscious processing in global workspace theory.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _ignition_reference():
    return float(next(s for s, a in zip(strengths, final) if a > 0.5))

In [ ]:
ignition_threshold = None  # your computation
check("Exercise 2", ignition_threshold, _ignition_reference())

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-11/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Set the credences in the lab to your own beliefs. Which architecture scores highest, and would you accept that verdict?
2. The simplified measure distinguishes loops from feedforward networks. Is that difference a plausible marker of consciousness? Why or why not?
3. Which theory of this week is most testable, and what experiment would you run on an AI system to test it?

## Weekly task and submission

Write about 600 words that assess one current AI system with the indicator method, using at least three theories, and discuss whether computational functionalism is a reasonable working assumption. Include the lab chart for your credences and attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Methods for assessing AI consciousness.** Compare the theory-heavy indicator method with approaches based on behaviour and with biological naturalism, and identify what each would count as evidence [3, 9, 10].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Baars, B. J. (1988). *A Cognitive Theory of Consciousness*. Cambridge University Press.

[2] Tononi, G. (2004). An information integration theory of consciousness. *BMC Neuroscience*, 5, 42. <https://doi.org/10.1186/1471-2202-5-42>

[3] Butlin, P., Long, R., Elmoznino, E., Bengio, Y., Birch, J., Constant, A., Deane, G., Fleming, S. M., Frith, C., Ji, X., Kanai, R., Klein, C., Lindsay, G., Michel, M., Mudrik, L., Peters, M. A. K., Schwitzgebel, E., Simon, J., & VanRullen, R. (2023). Consciousness in artificial intelligence: Insights from the science of consciousness. arXiv preprint arXiv:2308.08708. <https://arxiv.org/abs/2308.08708>

[4] Dehaene, S., & Naccache, L. (2001). Towards a cognitive neuroscience of consciousness: Basic evidence and a workspace framework. *Cognition*, 79(1-2), 1-37. <https://doi.org/10.1016/S0010-0277(00)00123-2>

[5] Tononi, G., Boly, M., Massimini, M., & Koch, C. (2016). Integrated information theory: From consciousness to its physical substrate. *Nature Reviews Neuroscience*, 17(7), 450-461. <https://doi.org/10.1038/nrn.2016.44>

[6] Albantakis, L., Barbosa, L., Findlay, G., et al. (2023). Integrated information theory (IIT) 4.0: Formulating the properties of phenomenal existence in physical terms. *PLOS Computational Biology*, 19(10), e1011465. <https://doi.org/10.1371/journal.pcbi.1011465>

[7] Rosenthal, D. M. (2005). *Consciousness and Mind*. Clarendon Press.

[8] Lamme, V. A. F. (2006). Towards a true neural stance on consciousness. *Trends in Cognitive Sciences*, 10(11), 494-501. <https://doi.org/10.1016/j.tics.2006.09.001>

[9] Chalmers, D. J. (2023). Could a large language model be conscious?. arXiv preprint arXiv:2303.07103. <https://arxiv.org/abs/2303.07103>

[10] Seth, A. K. (2025). Conscious artificial intelligence and biological naturalism. *Behavioral and Brain Sciences*. Published online 21 April 2025. <https://doi.org/10.1017/S0140525X25000032>